# News Forecasting — Model Consensus Analysis

Compare how frontier LLMs forecast real-world outcomes from news articles, with a focus on **model consensus**: which questions do all models agree on, and where do they diverge?

The full pipeline:
1. **Collect news** articles from a date range
2. **Generate questions** — binary yes/no forecasting questions about future events
3. **Find answers** — web search discovers what actually happened
4. **Render prompts** — format each question for model consumption
5. **Send to models** — multiple LLMs produce probability forecasts
6. **Score** — compare each model's forecasts against ground truth

In [12]:
%pip install lightningrod-ai python-dotenv pandas

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [13]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure the pipeline

The pipeline has six stages:

1. **NewsSeedGenerator** — collects news articles from a date range
2. **ForwardLookingQuestionGenerator** — creates binary yes/no forecasting questions from those articles
3. **WebSearchLabeler** — finds ground-truth answers by searching for what actually happened
4. **QuestionRenderer** — renders the final prompt with answer format instructions
5. **RolloutGenerator** — sends the rendered prompt to multiple LLMs for comparison
6. **RolloutScorer** — scores each model's probability forecast against the ground-truth label

We use dates a few months in the past so the questions can be resolved.

In [14]:
from datetime import datetime
from lightningrod import (
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    WebSearchLabeler,
    QuestionPipeline,
    NewsContextGenerator,
    QuestionRenderer,
    RolloutGenerator,
    RolloutScorer,
    BinaryAnswerType,
    open_router_model,
    lightningrod_model,
)

# Date range — adjust these to a period ~2-3 months in the past
START_DATE = datetime(2025, 11, 6)
END_DATE = datetime(2026, 3, 1)

seed_generator = NewsSeedGenerator(
    start_date=START_DATE,
    end_date=END_DATE,
    search_query="technology announcements",
)

answer_type = BinaryAnswerType()

question_generator = ForwardLookingQuestionGenerator(
    instructions="Generate forward-looking yes/no questions about tech announcements. "
    "Questions should be clearly resolvable within 1-2 months.",
    answer_type=answer_type,
)

labeler = WebSearchLabeler(answer_type=answer_type)

renderer = QuestionRenderer(answer_type=answer_type)

models = [
    open_router_model("openai/gpt-5.2"),
    open_router_model("anthropic/claude-sonnet-4.6"),
    open_router_model("google/gemini-3.1-pro-preview"),
    lightningrod_model("foresight-v3"),
]

context_generator = NewsContextGenerator()

rollout_generator = RolloutGenerator(models=models)

scorer = RolloutScorer(answer_type=answer_type)

pipeline = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    context_generators=[context_generator],
    labeler=labeler,
    renderer=renderer,
    rollout_generator=rollout_generator,
    scorer=scorer,
)

## Run the pipeline

> Note: This can take several minutes — the pipeline collects news, generates questions, searches for answers, then sends each question to all three models.

In [15]:
dataset = lr.transforms.run(pipeline, max_questions=600, name="News Forecasting Benchmark")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Warning                                                                                                     │
│                                                                                                                 │
│  Estimated cost ($60.33) exceeds current balance ($10.31). Consider adding credits before running this job.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

## View generated questions and labels

Each sample contains a forecasting question generated from a news article and a ground-truth label found via web search.

In [16]:
samples = dataset.download()

print(f"Generated {dataset.num_rows} samples ({dataset.valid_count() / dataset.num_rows * 100:.1f}% valid)\n")


Generated 120 samples (79.2% valid)



## Consensus analysis

Where do the models agree, and where do they diverge? `compute_consensus` extracts each model's predicted probability and computes:
- **spread** — max probability minus min probability across models (higher = more disagreement)
- **all_agree** — whether all models predict the same side of 0.5

In [17]:
from lightningrod.utils import compute_consensus
import pandas as pd

consensus = compute_consensus(samples)
n_agree = sum(1 for c in consensus if c["all_agree"])
n_total = len(consensus)

print(f"Consensus: {n_agree}/{n_total} questions have full agreement ({n_agree / n_total * 100:.0f}%)")
print(f"Disagreement: {n_total - n_agree}/{n_total} questions have models on opposite sides of 0.5")
print(f"Mean spread: {sum(c['spread'] for c in consensus) / n_total:.3f}")
print()

# Build a DataFrame with per-model predictions and spread
rows = []
for c in consensus:
    row = {"Question": c["question_text"], "Label": c["label"], "Spread": round(c["spread"], 3), "Agree": c["all_agree"]}
    for model, prob in c["predictions"].items():
        short_name = model.split("/")[-1] if "/" in model else model
        row[short_name] = round(prob, 3)
    rows.append(row)

df_consensus = pd.DataFrame(rows)
df_consensus

Consensus: 58/95 questions have full agreement (61%)
Disagreement: 37/95 questions have models on opposite sides of 0.5
Mean spread: 0.308



,Question,Label,Spread,Agree,gpt-5.2,claude-sonnet-4.6,gemini-3.1-pro-preview,foresight-v3
0,Will Apple Inc. officially announce a new gene...,0,0.93,False,0.08,0.95,0.02,0.17
1,Will Apple Inc. announce or release a new gene...,0,0.87,False,0.90,0.04,0.03,0.12
2,"Will the 'motorola signature' smartphone, anno...",1,0.76,False,0.55,0.88,0.85,0.12
3,Will the European Commission or European Parli...,0,0.70,False,0.18,0.35,0.88,0.22
4,"By January 15, 2026, will Supermicro appear in...",1,0.67,False,0.35,0.65,0.76,0.09
...,...,...,...,...,...,...,...,...
90,Will the Rwandan Ministry of ICT and Innovatio...,1,0.05,True,0.25,0.20,NaN,0.22
91,Will the solo exhibition of 'Calculating Empir...,1,0.05,True,0.93,0.90,0.95,0.93
92,Will the U.S. Attorney's Office for the Distri...,0,0.04,True,0.01,0.05,0.01,0.03
93,Will IBM publicly announce the deployment of a...,0,0.04,True,0.05,0.03,0.01,0.04


## Per-model accuracy (secondary)

For reference, here are the individual model metrics. `mean_reward` captures how well-calibrated each model's probability estimates are (higher is better).

In [18]:
from lightningrod.utils import compute_metrics_summary

summary = compute_metrics_summary(samples)
df_metrics = pd.DataFrame.from_dict(summary, orient="index")
df_metrics.index.name = "model"
df_metrics[["mean_reward", "parse_rate", "n_total"]]

,mean_reward,parse_rate,n_total
model,,,
openai/gpt-5.2,-0.230492,1.000000,95
anthropic/claude-sonnet-4.6,-0.222194,1.000000,95
google/gemini-3.1-pro-preview,-0.279008,0.978947,95
LightningRodLabs/foresight-v3,-0.298932,1.000000,95


## Next steps

- **Quick start**: See [01_quick_start.ipynb](../01_quick_start.ipynb) for the basic news forecasting pipeline
- **Document classification benchmark**: See [document_classification_benchmark.ipynb](document_classification_benchmark.ipynb) for benchmarking on a pre-existing dataset
- **Different question types**: See examples 04-07 for different question and answer types
- **Full API reference**: See [API.md](../../API.md) for all options and configurations